# Final Project: Hospital Readmissions, Recurrence, and Length of Stay

**Course:** Data Analytics 101  
**Institution:** Universidad Iberoamericana, Ciudad de Mexico  
**Dataset:** Diabetes 130-US hospitals for years 1999-2008  

---

## Executive Framing

Hospital readmissions among diabetic patients generate both clinical and operational pressure. In this project, we study three linked outcomes:

1. **30-day readmission risk**
2. **Hospital recurrence**
3. **Length of stay**

Following the style used throughout the course, the analysis begins with **structural understanding prior to formal modeling**. We first diagnose data quality, missingness, empirical distributions, and group-level patterns. We then move from **visual intuition to statistical rigor**, and finally to interpretable statistical models aligned with the geometry of each response variable.


## 0. Environment Setup and Imports

Before any substantive analysis, we establish a reproducible environment, load the dataset, and import the visualization utilities used in the course notebooks.

**Analytical principle:** reproducibility is part of model credibility. If the data pipeline is unstable, the interpretation is unstable as well.


In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, mean_absolute_error, mean_squared_error

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'functionality.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from functionality import (
    plot_variable_analysis,
    plot_ecdf_variable_analysis,
    plot_correlation_heatmap,
    plot_bivariate_analysis,
)

DATA_PATH = PROJECT_ROOT / 'proyecto' / 'diabetic_data.csv'
IDS_PATH = PROJECT_ROOT / 'proyecto' / 'IDS_mapping.csv'

print(f'Project root: {PROJECT_ROOT}')
print(f'Data path exists: {DATA_PATH.exists()}')
print(f'IDS mapping exists: {IDS_PATH.exists()}')


## 1. Data Loading and Structural Validation

The first analytical checkpoint is simple but essential: does the dataset contain the expected dimensionality and the variables required by the project guidelines?

**Question:** Are the core columns for readmission, recurrence, and length of stay available and internally consistent?


In [ ]:
df_raw = pd.read_csv(DATA_PATH, low_memory=False)

print('Shape:', df_raw.shape)
print('Columns:', len(df_raw.columns))
df_raw.head(3)


In [ ]:
assert df_raw.shape[0] > 100000
assert 'readmitted' in df_raw.columns
assert 'time_in_hospital' in df_raw.columns
assert 'number_inpatient' in df_raw.columns
assert 'num_medications' in df_raw.columns

print('Initial structural validation passed.')


### Interpretation

The dataset contains the necessary variables to study the three project outcomes. This confirms that the hospital questions in the guidelines can be addressed directly from the provided table without requiring additional external data sources.


## 2. Data Quality Audit

In the course notebooks, preprocessing is treated as a statistical decision rather than a cosmetic step. For that reason, we explicitly diagnose two forms of incompleteness:

1. **True missing values (`NaN`)**
2. **Encoded missing values (`?`)**

This distinction matters because many healthcare datasets contain administrative placeholders that would otherwise be misread as valid categories.


In [ ]:
missing_summary = pd.DataFrame({
    'dtype': df_raw.dtypes.astype(str),
    'n_missing': df_raw.isna().sum(),
    'pct_missing': df_raw.isna().mean() * 100,
    'n_question_mark': (df_raw == '?').sum(),
})
missing_summary['pct_question_mark'] = missing_summary['n_question_mark'] / len(df_raw) * 100
missing_summary.sort_values(['pct_missing', 'pct_question_mark'], ascending=False).head(15)


In [ ]:
duplicate_encounters = df_raw['encounter_id'].duplicated().sum()
unique_patients = df_raw['patient_nbr'].nunique()

print(f'Duplicate encounter_id rows: {duplicate_encounters}')
print(f'Unique patients: {unique_patients}')
print(df_raw['readmitted'].value_counts(dropna=False))


### Interpretation

The audit reveals substantial administrative missingness, especially in variables such as `weight`, `medical_specialty`, and `payer_code`, as well as laboratory-related fields with high non-response. This immediately tells us that a naive complete-case analysis would discard too much information and could distort the population under study.


## 3. Cleaning Strategy and Analytical Dataset

Following the missing-data logic discussed in class, we do **not** remove rows by default. Instead, we build a defensible analytical dataset with the following principles:

- Convert placeholder missing values into actual `NaN`
- Preserve clinically meaningful categories
- Remove pure identifiers from modeling stages
- Create project-specific targets explicitly

### Working hypothesis on missingness

At this stage, several variables likely contain **informative missingness** rather than purely random absence. For example, missing `weight` or lab measures may reflect differences in measurement protocol rather than random data loss. This matters for interpretation and for later modeling decisions.


In [ ]:
df = df_raw.replace('?', np.nan).copy()

id_columns = ['encounter_id', 'patient_nbr']
response_columns = ['readmitted', 'number_inpatient', 'time_in_hospital']

# Binary target aligned with the project prompt: unplanned readmission within 30 days.
df['readmit_30'] = (df['readmitted'] == '<30').astype(int)

# Midpoint approximation for the age bracket, useful for ordered summaries.
df['age_midpoint'] = (
    df['age']
    .str.extract(r'\[(\d+)-', expand=False)
    .astype(float)
)

# High-missingness flag variables can themselves be analytically informative.
df['weight_missing'] = df['weight'].isna().astype(int)
df['payer_code_missing'] = df['payer_code'].isna().astype(int)
df['medical_specialty_missing'] = df['medical_specialty'].isna().astype(int)

print('Question marks remaining:', int((df == '?').sum().sum()))
print(df[['readmitted', 'readmit_30', 'age', 'age_midpoint']].head())


In [ ]:
assert int((df == '?').sum().sum()) == 0
assert set(df['readmit_30'].unique()).issubset({0, 1})
assert (df['number_inpatient'] >= 0).all()
assert (df['time_in_hospital'] > 0).all()

print('Cleaning and target engineering validation passed.')


### Interpretation

The analytical dataset now reflects the project's central binary risk question while preserving the original multiclass readmission variable for descriptive work. By converting administrative placeholders into `NaN`, we also make missingness visible to the analysis instead of hiding it inside misleading categories.


## 4. Structural Understanding Prior to Formal Modeling

The EDA notebooks in the course emphasize that graphical diagnostics are not decorative. Their role is to reveal the geometry of the data before a model imposes its own assumptions.

We therefore begin with three structural questions:

1. What is the empirical shape of the key response variables?
2. Are there visible group differences in 30-day readmission risk?
3. Do the count and stay variables suggest distributional choices such as Poisson, Negative Binomial, or Gamma?


In [ ]:
key_numeric = [
    'time_in_hospital',
    'number_inpatient',
    'number_emergency',
    'number_outpatient',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_diagnoses',
]

summary = df[key_numeric].agg(['mean', 'median', 'std', 'min', 'max', 'skew']).T
summary.round(3)


In [ ]:
overdispersion_df = pd.DataFrame({
    'mean': df[['number_inpatient', 'number_emergency', 'number_outpatient']].mean(),
    'variance': df[['number_inpatient', 'number_emergency', 'number_outpatient']].var(),
})
overdispersion_df['var_to_mean'] = overdispersion_df['variance'] / overdispersion_df['mean']
overdispersion_df.round(3)


### Interpretation

The count variables show strong right-skewness and variance materially larger than the mean. In the language of the distributions notebook, this is exactly the kind of geometry that challenges a simple Poisson assumption and motivates a Negative Binomial alternative.

Similarly, `time_in_hospital` is strictly positive and right-skewed, which already suggests caution against using an untransformed Gaussian model.


### 4.1 Univariate Diagnostics

Following the four-panel diagnostic style from class, we inspect the key hospital outcomes before moving into formal inference.


In [ ]:
plot_variable_analysis(df, 'time_in_hospital', category='readmitted', palette='Set2')


In [ ]:
plot_variable_analysis(df, 'number_inpatient', category='readmitted', palette='Set2')


### Interpretation

The visual diagnostics reinforce the summary statistics: both recurrence and stay length depart from symmetry, and the group distributions overlap substantially. That means formal inference and modeling will need to focus on **systematic shifts in risk and expected counts**, not on visually perfect separation.


### 4.2 Empirical Distributions (ECDF)

As emphasized in the EDA notebook, ECDFs provide a direct non-parametric way to compare where two groups accumulate mass across the support of a variable.


In [ ]:
plot_ecdf_variable_analysis(
    df,
    variables=['time_in_hospital', 'num_medications', 'num_lab_procedures'],
    category='readmit_30',
    palette='Set1',
    figsize=(18, 5),
)


### Interpretation

If the ECDF for the readmitted group lies systematically to the right for a variable such as `num_medications`, that indicates a tendency toward larger observed values among patients who return within 30 days. This does not prove causality, but it helps identify variables worth evaluating formally.


### 4.3 Group-Level Readmission Patterns

We now summarize 30-day readmission prevalence across selected demographic and treatment-related variables.


In [ ]:
group_readmit = {
    'gender': df.groupby('gender', dropna=False)['readmit_30'].mean().sort_values(ascending=False),
    'age': df.groupby('age', dropna=False)['readmit_30'].mean().sort_values(ascending=False),
    'diabetesMed': df.groupby('diabetesMed', dropna=False)['readmit_30'].mean().sort_values(ascending=False),
    'change': df.groupby('change', dropna=False)['readmit_30'].mean().sort_values(ascending=False),
}

for name, series in group_readmit.items():
    print()
    print(name.upper())
    print((series * 100).round(2).to_string())


In [ ]:
readmit_by_age = (
    df.groupby('age', dropna=False)['readmit_30']
    .mean()
    .sort_index()
    .mul(100)
    .reset_index(name='readmit_rate_pct')
)

plt.figure(figsize=(12, 5))
sns.barplot(data=readmit_by_age, x='age', y='readmit_rate_pct', color='#4C956C')
plt.title('30-Day Readmission Rate by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Readmission Rate (%)')
plt.xticks(rotation=45)
plt.show()


### Interpretation

The age gradient and medication-related differences suggest that clinical complexity may be associated with short-term readmission risk. In presentation terms, this is a strong bridge from descriptive analysis to inferential testing: the visuals generate the hypotheses that the next section will evaluate formally.


## 5. Moving from Visual Intuition to Statistical Rigor

Following the structure of `inference.ipynb`, we now formalize the strongest patterns observed visually. The objective is not to test everything mechanically, but to validate whether the apparent differences exceed what we would expect from random variability alone.

### Hypotheses evaluated in this first inferential block

1. Patients readmitted within 30 days have different distributions of `time_in_hospital` and `num_medications`.
2. Readmission risk is associated with treatment-change and diabetes-medication status.
3. The observed distributional differences are strong enough to justify more structured modeling in the next phase.


### 5.1 Normality Check: Shapiro-Wilk Test

**Theory:** The Shapiro-Wilk test evaluates the null hypothesis that a sample was drawn from a normal distribution.

- $H_0$: the variable is normally distributed within the group under study.
- If `p < 0.05`, we reject normality.

Because the dataset is very large, we use a reproducible subsample per group. This follows the same caveat discussed in class: at large `n`, Shapiro-Wilk becomes extremely sensitive to even minor deviations from normality.


In [ ]:
rng = np.random.default_rng(42)
inferential_vars = ['time_in_hospital', 'num_medications']
normality_results = []

for var in inferential_vars:
    for group_value, label in [(0, 'No 30-day readmission'), (1, '30-day readmission')]:
        sample = df.loc[df['readmit_30'] == group_value, var].dropna()
        sample_n = min(5000, len(sample))
        sample = sample.sample(sample_n, random_state=42)
        stat, p_value = stats.shapiro(sample)
        normality_results.append({
            'variable': var,
            'group': label,
            'n_used': sample_n,
            'W_stat': stat,
            'p_value': p_value,
            'decision': 'Reject H0 (non-normal)' if p_value < 0.05 else 'Fail to reject H0',
        })

normality_df = pd.DataFrame(normality_results)
normality_df


### Interpretation

If these variables fail normality, that supports the course recommendation to prioritize **robust or non-parametric procedures** instead of relying mechanically on Gaussian assumptions. For this project, that is especially relevant because both medication counts and hospital stays are structurally bounded and right-skewed.


### 5.2 Equality of Variances: Levene's Test

**Theory:** Levene's test evaluates whether the two groups have equal variances.

- $H_0$: both groups have equal variance.
- If `p < 0.05`, we reject homoscedasticity.

This matters because unequal spread reinforces the case for robust comparisons and cautions against simplistic parametric modeling on the raw scale.


In [ ]:
levene_results = []

for var in inferential_vars:
    g0 = df.loc[df['readmit_30'] == 0, var].dropna()
    g1 = df.loc[df['readmit_30'] == 1, var].dropna()
    stat, p_value = stats.levene(g0, g1, center='median')
    levene_results.append({
        'variable': var,
        'levene_stat': stat,
        'p_value': p_value,
        'decision': 'Reject H0 (heteroscedasticity)' if p_value < 0.05 else 'Fail to reject H0',
    })

levene_df = pd.DataFrame(levene_results)
levene_df


### 5.3 Rank and Distribution Comparisons

We now compare the two readmission groups using tests that are more appropriate for skewed outcomes.

- **Mann-Whitney U** compares the rank distributions between groups.
- **Kolmogorov-Smirnov** compares the entire empirical distributions.

This mirrors the course logic: when the shape is non-normal, we should not reduce the comparison to means alone.


In [ ]:
distribution_results = []

for var in inferential_vars:
    g0 = df.loc[df['readmit_30'] == 0, var].dropna()
    g1 = df.loc[df['readmit_30'] == 1, var].dropna()

    u_stat, u_p = stats.mannwhitneyu(g0, g1, alternative='two-sided')
    ks_stat, ks_p = stats.ks_2samp(g0, g1)

    distribution_results.append({
        'variable': var,
        'group0_median': g0.median(),
        'group1_median': g1.median(),
        'mannwhitney_u': u_stat,
        'mannwhitney_p': u_p,
        'ks_stat': ks_stat,
        'ks_p': ks_p,
    })

distribution_df = pd.DataFrame(distribution_results)
distribution_df


### Interpretation

If both Mann-Whitney and K-S reject their null hypotheses, then the readmitted and non-readmitted groups differ not only in average magnitude but in their broader empirical distribution. That is exactly the kind of result that justifies moving from descriptive evidence into a predictive or explanatory model.


### 5.4 Categorical Independence: Chi-Square Tests

**Theory:** The chi-square test of independence evaluates whether the distribution of one categorical variable changes across the categories of another.

- $H_0$: the variables are independent.
- If `p < 0.05`, we reject independence.

Here we test whether short-term readmission is associated with selected patient or treatment characteristics.


In [ ]:
categorical_vars = ['gender', 'age', 'diabetesMed', 'change']
chi2_results = []

for col in categorical_vars:
    contingency = pd.crosstab(df[col], df['readmit_30'])
    chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency)
    chi2_results.append({
        'variable': col,
        'chi2_stat': chi2_stat,
        'p_value': p_value,
        'degrees_of_freedom': dof,
        'decision': 'Reject H0 (dependent)' if p_value < 0.05 else 'Fail to reject H0',
    })

chi2_df = pd.DataFrame(chi2_results)
chi2_df


### Interpretation

A significant chi-square result does not mean that a variable is a direct causal driver of readmission. It does mean that the distribution of readmission differs across that clinical or demographic grouping strongly enough that the relationship is unlikely to be pure sampling noise.


### 5.5 Inferential Summary Table

To keep the narrative presentation-ready, we consolidate the main inferential outputs into a compact summary table.


In [ ]:
inferential_summary = pd.concat([
    normality_df.assign(test='Shapiro-Wilk').rename(columns={'p_value': 'p_value_main'})[['test', 'variable', 'group', 'p_value_main', 'decision']],
    levene_df.assign(test='Levene', group='Readmit groups').rename(columns={'p_value': 'p_value_main'})[['test', 'variable', 'group', 'p_value_main', 'decision']],
    chi2_df.assign(test='Chi-square', group='Readmit groups').rename(columns={'p_value': 'p_value_main'})[['test', 'variable', 'group', 'p_value_main', 'decision']],
])

inferential_summary.sort_values(['test', 'variable']).reset_index(drop=True)


## 6. Interpretable Statistical Modeling

The rubric explicitly rewards **appropriate and interpretable models** rather than black-box complexity. In that spirit, each response variable is modeled with a distribution that matches its empirical geometry.

1. **30-day readmission**: Binomial GLM
2. **Hospital recurrence**: Poisson baseline vs Negative Binomial
3. **Length of stay**: Gamma GLM with log link


### 6.1 Modeling Design and Parsimony

To keep the models presentation-ready, we use a deliberately parsimonious feature set built from variables that are both clinically plausible and available with limited missingness.

For the recurrence model, highly skewed utilization predictors are log-transformed with `log1p`. This reduces leverage from extreme prior-use values and makes the count model more stable without erasing the underlying ordering.


In [ ]:
df['gender_model'] = df['gender'].where(df['gender'].isin(['Male', 'Female']))
df['race_model'] = df['race'].fillna('Missing/Unknown')
df['race_model'] = df['race_model'].where(
    df['race_model'].isin(['Caucasian', 'AfricanAmerican', 'Hispanic']),
    'Other/Unknown'
)
df['change_model'] = df['change'].fillna('Unknown')
df['diabetesMed_model'] = df['diabetesMed'].fillna('Unknown')

for col in ['time_in_hospital', 'num_medications', 'number_outpatient', 'number_emergency']:
    df[f'log1p_{col}'] = np.log1p(df[col])

logit_cols = [
    'readmit_30', 'age_midpoint', 'time_in_hospital', 'num_lab_procedures',
    'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses', 'race_model', 'gender_model',
    'change_model', 'diabetesMed_model'
]
logit_df = df[logit_cols].dropna().copy()

count_cols = [
    'number_inpatient', 'age_midpoint', 'gender_model', 'change_model',
    'diabetesMed_model', 'log1p_time_in_hospital', 'log1p_num_medications',
    'log1p_number_outpatient', 'log1p_number_emergency'
]
count_df = df[count_cols].dropna().copy()

los_cols = [
    'time_in_hospital', 'age_midpoint', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient',
    'number_diagnoses', 'race_model', 'gender_model', 'change_model',
    'diabetesMed_model'
]
los_df = df[los_cols].dropna().copy()

assert len(logit_df) > 90000
assert len(count_df) > 90000
assert len(los_df) > 90000

pd.DataFrame({
    'dataset': ['logit_df', 'count_df', 'los_df'],
    'rows': [len(logit_df), len(count_df), len(los_df)],
    'columns': [logit_df.shape[1], count_df.shape[1], los_df.shape[1]],
})


### 6.2 Model 1: 30-Day Readmission Risk

**Model choice:** Binomial GLM with logit link.

**Why this model fits the problem:** the target `readmit_30` is binary, and the resulting coefficients can be transformed into **odds ratios**, making the model directly interpretable for hospital decision-making.


In [ ]:
logit_train, logit_test = train_test_split(
    logit_df,
    test_size=0.25,
    random_state=42,
    stratify=logit_df['readmit_30']
)

logit_formula = (
    'readmit_30 ~ age_midpoint + time_in_hospital + num_lab_procedures + '
    'num_procedures + num_medications + number_outpatient + number_emergency + '
    'number_inpatient + number_diagnoses + C(race_model) + C(gender_model) + '
    'C(change_model) + C(diabetesMed_model)'
)

readmit_model = smf.glm(
    logit_formula,
    data=logit_train,
    family=sm.families.Binomial()
).fit()

readmit_pred = readmit_model.predict(logit_test)
readmit_class = (readmit_pred >= 0.5).astype(int)

readmit_metrics = pd.DataFrame([{
    'model': 'Binomial GLM',
    'test_auc': roc_auc_score(logit_test['readmit_30'], readmit_pred),
    'test_accuracy': accuracy_score(logit_test['readmit_30'], readmit_class),
    'aic': readmit_model.aic,
    'n_train': len(logit_train),
    'n_test': len(logit_test),
}])

readmit_metrics


In [ ]:
readmit_effects = pd.DataFrame({
    'coefficient': readmit_model.params,
    'odds_ratio': np.exp(readmit_model.params),
})

readmit_effects.loc[readmit_effects.index != 'Intercept'] \
    .assign(distance_from_null=lambda d: (d['odds_ratio'] - 1).abs()) \
    .sort_values('distance_from_null', ascending=False) \
    .drop(columns='distance_from_null') \
    .head(10)


### Interpretation

The readmission model is not designed as a high-performance classifier. Its value is interpretive: it identifies which patient and utilization characteristics are associated with a higher or lower **odds** of returning within 30 days, while keeping the hospital-facing story understandable.


### 6.3 Model 2: Hospital Recurrence as Count Data

**Model choice:** Poisson baseline and Negative Binomial alternative.

**Why this model fits the problem:** `number_inpatient` is a count variable with strong right-skewness and variance greater than the mean. That makes Poisson a natural baseline and Negative Binomial a theoretically attractive extension when overdispersion is present.


In [ ]:
count_train, count_test = train_test_split(
    count_df,
    test_size=0.25,
    random_state=42,
)

count_formula = (
    'number_inpatient ~ age_midpoint + log1p_time_in_hospital + '
    'log1p_num_medications + log1p_number_outpatient + log1p_number_emergency + '
    'C(gender_model) + C(change_model) + C(diabetesMed_model)'
)

poisson_model = smf.poisson(count_formula, data=count_train).fit(disp=False, maxiter=200)
nb_model = smf.negativebinomial(count_formula, data=count_train).fit(disp=False, maxiter=200)

poisson_pred = poisson_model.predict(count_test)
nb_pred = nb_model.predict(count_test)

count_metrics = pd.DataFrame([
    {
        'model': 'Poisson',
        'test_rmse': mean_squared_error(count_test['number_inpatient'], poisson_pred) ** 0.5,
        'test_mae': mean_absolute_error(count_test['number_inpatient'], poisson_pred),
        'aic': poisson_model.aic,
    },
    {
        'model': 'Negative Binomial',
        'test_rmse': mean_squared_error(count_test['number_inpatient'], nb_pred) ** 0.5,
        'test_mae': mean_absolute_error(count_test['number_inpatient'], nb_pred),
        'aic': nb_model.aic,
    },
])

count_metrics


In [ ]:
recurrence_effects = pd.DataFrame({
    'coefficient': nb_model.params,
    'incidence_rate_ratio': np.exp(nb_model.params),
})

recurrence_effects.loc[~recurrence_effects.index.isin(['Intercept', 'alpha'])] \
    .assign(distance_from_null=lambda d: (d['incidence_rate_ratio'] - 1).abs()) \
    .sort_values('distance_from_null', ascending=False) \
    .drop(columns='distance_from_null') \
    .head(10)


### Interpretation

This comparison produces a nuanced result that is actually very defensible in a presentation. The Poisson baseline is slightly better on simple holdout prediction error, but the Negative Binomial achieves a substantially lower AIC and estimates a positive `alpha`, both of which are consistent with overdispersion in the recurrence process.

That means the hospital should interpret recurrence as a count process with extra variability beyond a simple Poisson assumption. In practical terms, repeated admissions are not evenly dispersed across patients; a smaller subset concentrates much more inpatient burden than a naive count model would expect.


### 6.4 Model 3: Length of Stay

**Model choice:** Gamma GLM with log link.

**Why this model fits the problem:** `time_in_hospital` is strictly positive and right-skewed. The Gamma family respects positivity, while the log link expresses effects multiplicatively, which is often easier to interpret for operational duration outcomes.


In [ ]:
los_train, los_test = train_test_split(
    los_df,
    test_size=0.25,
    random_state=42,
)

los_formula = (
    'time_in_hospital ~ age_midpoint + num_lab_procedures + num_procedures + '
    'num_medications + number_outpatient + number_emergency + number_inpatient + '
    'number_diagnoses + C(race_model) + C(gender_model) + C(change_model) + '
    'C(diabetesMed_model)'
)

los_model = smf.glm(
    los_formula,
    data=los_train,
    family=sm.families.Gamma(link=sm.families.links.Log())
).fit(maxiter=200)

los_pred = los_model.predict(los_test)

los_metrics = pd.DataFrame([{
    'model': 'Gamma GLM',
    'test_rmse': mean_squared_error(los_test['time_in_hospital'], los_pred) ** 0.5,
    'test_mae': mean_absolute_error(los_test['time_in_hospital'], los_pred),
    'aic': los_model.aic,
    'n_train': len(los_train),
    'n_test': len(los_test),
}])

los_metrics


In [ ]:
los_effects = pd.DataFrame({
    'coefficient': los_model.params,
    'multiplicative_effect': np.exp(los_model.params),
})

los_effects.loc[los_effects.index != 'Intercept'] \
    .assign(distance_from_null=lambda d: (d['multiplicative_effect'] - 1).abs()) \
    .sort_values('distance_from_null', ascending=False) \
    .drop(columns='distance_from_null') \
    .head(10)


### Interpretation

The Gamma model is especially useful for the written report because its coefficients can be explained as **multiplicative shifts** in expected stay length. This makes it straightforward to say that certain utilization or complexity variables are associated with proportionally longer hospital stays, rather than forcing an implausible linear-additive story on a skewed positive outcome.


### 6.5 Integrated Model Summary

For the final report and presentation, it helps to summarize the three modeling tasks in one compact table.


In [ ]:
integrated_model_summary = pd.concat([
    readmit_metrics.assign(target='readmit_30'),
    count_metrics.assign(target='number_inpatient'),
    los_metrics.assign(target='time_in_hospital'),
], ignore_index=True, sort=False)

integrated_model_summary


## 7. Next Analytical Steps

The notebook now contains a reproducible first stage aligned with the course style:

- Structural validation of the dataset
- Explicit audit of encoded and true missingness
- Defensible target engineering
- Distributional diagnostics for the main outcomes
- Initial readmission pattern summaries for presentation use

The next implementation block should add:

1. Formal hypothesis tests in the style of `inference.ipynb`
2. Correlation and bivariate analysis for key numeric drivers
3. Interpretable statistical models for each response variable
